# display
> display helpers

In [ ]:
#| default_exp display

In [ ]:
#| export
from pathlib import Path
from typing import Mapping, Sequence
from fastcore.foundation import AttrDict, patch
from fastcore.xml import FT
from fasthtml.components import Div, Details, Summary, Ul, Li, Span, show
from fasthtml.xtend import Style
from dialoghelper.core import add_html
from pote.common import is_listy, flatten, shorten

In [ ]:
import json
from typing import Any
from typing import Literal
from functools import partial
from pathlib import Path
from typing import cast
from IPython.display import HTML

import fastcore.all as FC
from fastcore.foundation import noop, patch
from fastcore.xml import to_xml
from fastcore.test import *

from pote.common import *
from pote.dutil import link_msg

In [ ]:
#| export
_n = '\n'

## ansi color codes
> Helpers and utils to work with ansi codes in notebooks.

What does the above bash script do?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

It's a comprehensive **ANSI escape code generator** for bash terminals. It provides a library of functions (and a CLI) to control terminal output styling and behaviour, including:

- **Text formatting** — bold, faint, italic, underline, blink, strikethrough, inverse, invisible, etc.
- **Foreground & background colours** — standard 8 colours, "intense" variants, 256-colour mode, and full RGB (24-bit)
- **Cursor control** — move up/down/forward/backward, save/restore position, show/hide cursor, jump to specific row/column
- **Display manipulation** — erase lines/display, insert/delete lines or characters, scroll up/down
- **Terminal reporting** — query cursor position, window size (chars and pixels), window state, title, icon name (requires terminal support, reads back from stdin)
- **Miscellaneous** — set window title/icon, ring the bell, print colour tables/code charts, full terminal reset

It can be **sourced** as a library (to use `ansi::*` functions directly in other scripts) or **run directly** as a CLI tool where options like `--bold --red "Hello"` emit the appropriate escape sequences around the text, then automatically restore the terminal state afterward (unless `--no-restore` is passed).

<details class='token-usage-details'><summary>$0.1282</summary>

`total=24,593 | in=24,292 | out=301 | cached=0.0% | cache_new=10,747 | $0.1282`

</details>


Is there something similar directly in Python, stdlib or external package?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, several options:

**stdlib:**
- `curses` — full terminal control (cursor, colors, input), but complex and low-level

**External packages (much more practical):**
- **`rich`** — the most popular; pretty printing, colored text, tables, progress bars, markdown rendering in terminal, syntax highlighting
- **`colorama`** — simple cross-platform ANSI color support (especially useful on Windows)
- **`blessed`** / **`urwid`** — full curses-like terminal control in a friendlier API
- **`prompt_toolkit`** — advanced terminal UIs, used by IPython/Jupyter under the hood

For most use cases, **`rich`** is the go-to choice — it covers the vast majority of what that bash script does (colors, styles, cursor control via its `Console` and `Live` APIs) and is widely used and well-maintained.

<details class='token-usage-details'><summary>$0.0883</summary>

`total=24,904 | in=24,663 | out=241 | cached=43.6% | cache_new=13,720 | $0.0883`

</details>


I know about `rich` and it's pre-installed here in Solveit, but the problem is it converts ansi codes to HTML in Solveit (or general Jupyter Notebooks) so the output is huge. I want pure ansi codes, SolveIt outputs directly support them.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

For pure ANSI strings with no rendering layer, the lightest options are:

- **`termcolor`** — very simple, just wraps strings with ANSI codes via `colored()`/`cprint()`
- **`sty`** — composable style objects (e.g. `fg.red + bg.blue + ef.bold`), pure strings, no magic
- **Raw codes directly** — since you just need `\033[...m` sequences, a tiny hand-rolled helper is often enough and has zero dependencies

`sty` is the nicest of the packages for this use case since styles are just string prefixes you compose freely. Would you like to go with that, or roll a minimal helper?

<details class='token-usage-details'><summary>$0.1485</summary>

`total=25,356 | in=24,991 | out=365 | cached=0.0% | cache_new=24,746 | $0.1485`

</details>


Ok, for now I just need the table of all ansi colors. Please, replicate in Python the `ansi::colorTable()` function from the bash script above.

In [ ]:
#| export
ESC = '\033['
def sgr(*c): return f"{ESC}{';'.join(map(str,c))}m"

def color_table():
    fgs   = [30,31,32,33,34,35,36,37]
    fgs_i = [90,91,92,93,94,95,96,97]
    bgs = [('(none)',49),('bg-black',40),('+ intense',100),('bg-red',41),('+ intense',101),
           ('bg-green',42),('+ intense',102),('bg-yellow',43),('+ intense',103),
           ('bg-blue',44),('+ intense',104),('bg-magenta',45),('+ intense',105),
           ('bg-cyan',46),('+ intense',106),('bg-white',47),('+ intense',107)]
    def cell(fg, fg_i, bg):
        return (sgr(bg,fg)+sgr(2)+'f'+sgr(22)+'n'+sgr(1)+'b'+sgr(22) +
                sgr(fg_i)+sgr(2)+'F'+sgr(22)+'N'+sgr(1)+'B'+sgr(22)+sgr(0))
    print(f"{'':12} black   red  green  yellow  blue  magenta cyan   white")
    for lbl, bg in bgs:
        print(f'{lbl:<12}' + ' '.join(cell(fg, fg_i, bg) for fg,fg_i in zip(fgs,fgs_i)))

In [ ]:
color_table()

             black   red  green  yellow  blue  magenta cyan   white
(none)      fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB
bg-black    fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB
+ intense   fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB
bg-red      fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB
+ intense   fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB
bg-green    fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB
+ intense   fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB
bg-yellow   fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB
+ intense   fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB
bg-blue     fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB
+ intense   fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB
bg-magenta  fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB
+ intense   fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB
bg-cyan     fnbFNB fnbFNB fnbFNB fnbFNB fnbFNB f

Ok, now same with `ansi::colorCodes()` function to get the actual codes.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The ANSI codes are being stripped in cell output. Looking at the bash output, each "patch" is just a colored background block showing the code number. Here's the Python version:

In [ ]:
#| export
def color_codes():
    def patch(code): return f"{sgr(48,5,code)} {code:3} {sgr(49)}"
    
    print('Standard: ' + sgr(1) + sgr(37) + ''.join(patch(c) + (sgr(30) if c==6 else '') for c in range(8)) + sgr(0))
    print('Intense:  ' + sgr(37) + ''.join(patch(c) + (sgr(30) if c==8 else '') for c in range(8,16)) + sgr(0))
    print()
    for i in [16, 22, 28]:
        for j in [i, i+36, i+72, i+108, i+144, i+180]:
            print(sgr(1,37) + ''.join(patch(c) for c in range(j,j+6)) + sgr(0) + '    ' +
                  sgr(30) + ''.join(patch(c) for c in range(j+18,j+24)) + sgr(0))
        print()
    print('Grays:    ' + sgr(1,37) + ''.join(patch(c) for c in range(232,244)) + sgr(0))
    print('          ' + sgr(30)   + ''.join(patch(c) for c in range(244,256)) + sgr(0))

In [ ]:
color_codes()

Standard:    0    1    2    3    4    5    6    7 
Intense:     8    9   10   11   12   13   14   15 

  16   17   18   19   20   21       34   35   36   37   38   39 
  52   53   54   55   56   57       70   71   72   73   74   75 
  88   89   90   91   92   93      106  107  108  109  110  111 
 124  125  126  127  128  129      142  143  144  145  146  147 
 160  161  162  163  164  165      178  179  180  181  182  183 
 196  197  198  199  200  201      214  215  216  217  218  219 

  22   23   24   25   26   27       40   41   42   43   44   45 
  58   59   60   61   62   63       76   77   78   79   80   81 
  94   95   96   97   98   99      112  113  114  115  116  117 
 130  131  132  133  134  135      148  149  150  151  152  153 
 166  167  168  169  170  171      184  185  186  187  188  189 
 202  203  204  205  206  207      220  221  222  223  224  225 

  28   29   30   31   32   33       46   47   48   49   50   51 
  64   65   66   67   68   69       82   83   84  

In [ ]:
for code in range(256):
    print(f'\033[38;5;{code}m\\033[38;5;{code}m\033[0m')

\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033
\033


In [ ]:
code = 250
print(f"\033[38;5;{code}m" + 'hello world' + '\033[0m')

hello world


In [ ]:
code = 34
print(f"\033[{code}m" + 'hello world' + '\033[0m')

hello world


## DetailsTree
> Collapsible tree viewer component for notebooks

A FastHTML component that renders trees as expandable `<details>` elements.

In [ ]:
tr = {  # => div
        'Apollo astronauts': [  # ul
            'Neil Armstrong',  # li
            'Alan Bean',  # li
            {  # li
                'other': [  # ul
                    'Bruce Wayne',  # li
                    'Clark Kent',  # li
                    'Peter Parker'  # li
                ]
            },
            'Edgar Mitchell',   # li
            'Alan Shepard'  # li
        ],
        'Apollo 11': [  # ul
            'Neil Armstrong',   # li
            'Buzz Aldrin',   # li
            'Edgar Mitchell',   # li
            {  # li
                'a':  # ul
                    1  # li
            },
            'Alan Shepard'  # li
        ]
}

In [ ]:
css = '\n'.join((await read_msg()).content.splitlines()[1:])
print(Style(css))

<style>.tree ul { list-style-type:none; list-style-position: outside; padding-inline-start: 22px; margin: 0px; }</style>


In [ ]:
#| export
def add_style(css:str):
    _stl = Div(Style(css), hx_swap_oob='beforeend:#dialog-container')
    add_html(_stl)

In [ ]:
add_style(css)

In [ ]:
%%html
<div class="tree">
<ul data-tree-name="Apollo astronauts">
  <li data-it>Neil Armstrong</li>
  <li data-it><a href="/asd">Alan Bean</a></li>
  <li data-it>other<ul data-tree-name="other">
    <li data-it>Bruce Wayne</li>
    <li data-it>Clark Kent</li>
    <li data-it>Peter Parker</li>
  </ul></li>
  <li data-it>Edgar Mitchell</li>
  <li data-it>Alan Shepard</li>
</ul>

<ul data-tree-name="Apollo 11">
  <li data-it>Neil Armstrong </li>
  <li data-it>Buzz Aldrin</li>
  <li data-it>Edgar Mitchell</li>
  <li data-it>a<ul data-tree-name="a">
    <li data-it>1</li>
  </ul></li>
  <li data-it>Alan Shepard</li>
</ul>
</div>

HTML(<div class="tree">
<ul data-tree-name="Apollo astronauts">
  <li data-it>Neil Armstrong</li>
  <li data-it><a href="/asd">Alan Bean</a></li>
  <li data-it>other<ul data-tree-name="other">
    <li data-it>Bruce Wayne</li>
    <li data-it>Clark Kent</li>
    <li data-it>Peter Parker</li>
  </ul></li>
  <li data-it>Edgar Mitchell</li>
  <li data-it>Alan Shepard</li>
</ul>

<ul data-tree-name="Apollo 11">
  <li data-it>Neil Armstrong </li>
  <li data-it>Buzz Aldrin</li>
  <li data-it>Edgar Mitchell</li>
  <li data-it>a<ul data-tree-name="a">
    <li data-it>1</li>
  </ul></li>
  <li data-it>Alan Shepard</li>
</ul>
</div>
)

In [ ]:
#| exporti
def it2lbl(it, nm=''):
    # return ((nm, it) if nm else (it,)) if isinstance(it, FT) else (str(it),)
    it = it if isinstance(it, FT) else str(it)
    return (nm, it) if nm else (it,)

In [ ]:
#| export
def list2lis(its:list, dproc, eproc=it2lbl) -> list[Li]:
    return flatten([
        [Li(data_it=True)(*eproc(ul,k)) for k,ul in dproc(it).items()] if isinstance(it, dict) else
        Li(data_it=True)(*eproc(it))
        for it in its], (FT,))

def dict2uls(d:dict, eproc=it2lbl) -> dict[Ul]:
    return {
        k:Ul(data_coll_name=k)(*list2lis(v if is_listy(v) else [v], dict2uls, eproc)) 
        for k,v in d.items()
    }

In [ ]:
uls = dict2uls(tr)
print(_ := to_xml(Div(*uls.values(), cls='tree')))
HTML(_)

<div class="tree">
  <ul data-coll-name="Apollo astronauts">
    <li data-it>Neil Armstrong</li>
    <li data-it>Alan Bean</li>
    <li data-it>
other      <ul data-coll-name="other">
        <li data-it>Bruce Wayne</li>
        <li data-it>Clark Kent</li>
        <li data-it>Peter Parker</li>
      </ul>
    </li>
    <li data-it>Edgar Mitchell</li>
    <li data-it>Alan Shepard</li>
  </ul>
  <ul data-coll-name="Apollo 11">
    <li data-it>Neil Armstrong</li>
    <li data-it>Buzz Aldrin</li>
    <li data-it>Edgar Mitchell</li>
    <li data-it>
a      <ul data-coll-name="a">
        <li data-it>1</li>
      </ul>
    </li>
    <li data-it>Alan Shepard</li>
  </ul>
</div>



HTML(<div class="tree">
  <ul data-coll-name="Apollo astronauts">
    <li data-it>Neil Armstrong</li>
    <li data-it>Alan Bean</li>
    <li data-it>
other      <ul data-coll-name="other">
        <li data-it>Bruce Wayne</li>
        <li data-it>Clark Kent</li>
        <li data-it>Peter Parker</li>
      </ul>
    </li>
    <li data-it>Edgar Mitchell</li>
    <li data-it>Alan Shepard</li>
  </ul>
  <ul data-coll-name="Apollo 11">
    <li data-it>Neil Armstrong</li>
    <li data-it>Buzz Aldrin</li>
    <li data-it>Edgar Mitchell</li>
    <li data-it>
a      <ul data-coll-name="a">
        <li data-it>1</li>
      </ul>
    </li>
    <li data-it>Alan Shepard</li>
  </ul>
</div>
)

In [ ]:
from pote.tree import transform, pprint

In [ ]:
tr

{'Apollo astronauts': ['Neil Armstrong',
  'Alan Bean',
  {'other': ['Bruce Wayne', 'Clark Kent', 'Peter Parker']},
  'Edgar Mitchell',
  'Alan Shepard'],
 'Apollo 11': ['Neil Armstrong',
  'Buzz Aldrin',
  'Edgar Mitchell',
  {'a': 1},
  'Alan Shepard']}

In [ ]:
pprint(tr, style='tree')

Apollo astronauts
├─ Neil Armstrong
├─ Alan Bean
├─ other
│  ├─ Bruce Wayne
│  ├─ Clark Kent
│  └─ Peter Parker
├─ Edgar Mitchell
└─ Alan Shepard
Apollo 11
├─ Neil Armstrong
├─ Buzz Aldrin
├─ Edgar Mitchell
├─ a
│  └─ 1
└─ Alan Shepard


In [ ]:
def _details_transform():
    def _leaf(p, v, ctx): return Li(data_it=True)(v)
    def _node(p, v, ctx):
        wrp, nm = (Li(data_it=True), next(iter(v))) if p else (noop, next(iter(v)))
        return wrp(nm, Ul(*next(iter(v.values()))))
    return dict(on_leaf=_leaf, on_node=_node)

def show_tree(tree):
    show(transform(tr, **_details_transform()))

In [ ]:
_ = transform(tr, **_details_transform())
print(_)

Apollo astronauts


In [ ]:
#| exporti
def it2lbl(it, nm=''):
    # return ((nm, it) if nm else (it,)) if isinstance(it, FT) else (str(it),)
    it = it if isinstance(it, FT) else str(it)
    return (nm, it) if nm else (it,)

In [ ]:
#| export
def list2lis(its:list, dproc, eproc=it2lbl) -> list[Li]:
    return flatten([
        [Li(data_it=True)(*eproc(ul,k)) for k,ul in dproc(it).items()] if isinstance(it, dict) else
        Li(data_it=True)(*eproc(it))
        for it in its], (FT,))

def dict2uls(d:dict, eproc=it2lbl) -> dict[Ul]:
    return {
        k:Ul(data_coll_name=k)(*list2lis(v if is_listy(v) else [v], dict2uls, eproc)) 
        for k,v in d.items()
    }

In [ ]:
uls = dict2uls(tr)
print(_ := to_xml(Div(*uls.values(), cls='tree')))
HTML(_)

In [ ]:
%%html
<div class="coll">
<details open><summary>Apollo astronauts</summary><ul data-coll-name="Apollo astronauts">
  <li data-it>Neil Armstrong</li>
  <li data-it><a href="/asd">Alan Bean</a></li>
  <li data-it><details open><summary>other</summary><ul data-coll-name="other">
    <li data-it>Bruce Wayne</li>
    <li data-it>Clark Kent</li>
    <li data-it>Peter Parker</li>
  </ul></details></li>
  <li data-it>Edgar Mitchell</li>
  <li data-it>Alan Shepard</li>
</ul></details>

<details open><summary>Apollo 11</summary><ul data-coll-name="Apollo 11">
  <li data-it>Neil Armstrong </li>
  <li data-it>Buzz Aldrin</li>
  <li data-it>Edgar Mitchell</li>
  <li data-it><details open><summary>a</summary><ul data-coll-name="a">
    <li data-it>1</li>
  </ul></details></li>
  <li data-it>Alan Shepard</li>
</ul></details>
</div>

HTML(<div class="coll">
<details open><summary>Apollo astronauts</summary><ul data-coll-name="Apollo astronauts">
  <li data-it>Neil Armstrong</li>
  <li data-it><a href="/asd">Alan Bean</a></li>
  <li data-it><details open><summary>other</summary><ul data-coll-name="other">
    <li data-it>Bruce Wayne</li>
    <li data-it>Clark Kent</li>
    <li data-it>Peter Parker</li>
  </ul></details></li>
  <li data-it>Edgar Mitchell</li>
  <li data-it>Alan Shepard</li>
</ul></details>

<details open><summary>Apollo 11</summary><ul data-coll-name="Apollo 11">
  <li data-it>Neil Armstrong </li>
  <li data-it>Buzz Aldrin</li>
  <li data-it>Edgar Mitchell</li>
  <li data-it><details open><summary>a</summary><ul data-coll-name="a">
    <li data-it>1</li>
  </ul></details></li>
  <li data-it>Alan Shepard</li>
</ul></details>
</div>
)

In [ ]:
#| exporti
def it2lbl2(it, nm=''):
    return (it,) if isinstance(it, FT) else (str(it),)

In [ ]:
#| export
def dict2dtls(d:dict, eproc=it2lbl2) -> dict[Details]:
    return {
        k:Details(open=True, data_coll_name=k)(
            Summary(k),
            Ul()(*list2lis(v if is_listy(v) else [v], dict2dtls, eproc))
        )
        for k,v in d.items()
    }

In [ ]:
uls = dict2dtls(coll)
print(_ := to_xml(Div(*uls.values(), cls='coll')))
HTML(_)

In [ ]:
def _v2tag(v):
    "Render value with appropriate CSS class based on type"
    c = 'None' if v is None else ''
    return Span(shorten(v, 'r', 140) if v is not None else 'None', cls=f"v{' '+c if c else ''}")

In [ ]:

test_eq(str(_v2tag(None)), '<span class="v None">None</span>')
test_eq(str(_v2tag(1)), '<span class="v">1</span>')

In [ ]:
_ = '\n'.join((await read_msg()).content.splitlines()[1:])
await link_msg(f"_css_coll = '''\n{_}\n'''", msg_type='code', is_exported=True);

In [ ]:
#| export
_css_coll = '''
.coll ul { list-style-type:none; list-style-position: outside; padding-inline-start: 22px; margin: 0px; }
.coll .None { color: gray; }
'''
#| linkedto: _f292d92c

In [ ]:
#| export
class DetailsColl(dict):
    "Interactive collapsible collection viewer with HTML details/summary structure"
    def __init__(self, d, summary:str='', open:bool=True, openall:bool=False):
        super().__init__(d or {})
        self.summary, self.open, self.openall = str(summary), open, openall
    def show(self): show(self)
    def __ft__(self, its=None, summary:str|None=None, lvl:int=0, open:bool=False):
        return Div(*dict2dtls(self).values(), cls='coll')
    _css_ = _css_coll
    @classmethod
    def setup_style(cls): add_style(cls._css_)

In [ ]:
# DetailsColl.setup_style()

In [ ]:
dtl = DetailsColl({'a':1})
print(to_xml(dtl))
dtl.show()

<div class="coll">
<details open data-coll-name="a"><summary>a</summary>    <ul>
      <li data-it>1</li>
    </ul>
</details></div>



HTML(<div class="coll">
<details open data-coll-name="a"><summary>a</summary>    <ul>
      <li data-it>1</li>
    </ul>
</details></div>
)

In [ ]:
dtl = DetailsColl({'a':1, 'b':2})
print(to_xml(dtl))
dtl.show()

<div class="coll">
<details open data-coll-name="a"><summary>a</summary>    <ul>
      <li data-it>1</li>
    </ul>
</details><details open data-coll-name="b"><summary>b</summary>    <ul>
      <li data-it>2</li>
    </ul>
</details></div>



HTML(<div class="coll">
<details open data-coll-name="a"><summary>a</summary>    <ul>
      <li data-it>1</li>
    </ul>
</details><details open data-coll-name="b"><summary>b</summary>    <ul>
      <li data-it>2</li>
    </ul>
</details></div>
)

In [ ]:
dtl = DetailsColl({'a':['abc', 13, {'d':['efg', 133]}, None]})
print(to_xml(dtl))
dtl.show()

<div class="coll">
<details open data-coll-name="a"><summary>a</summary>    <ul>
      <li data-it>abc</li>
      <li data-it>13</li>
      <li data-it>
<details open data-coll-name="d"><summary>d</summary>          <ul>
            <li data-it>efg</li>
            <li data-it>133</li>
          </ul>
</details>      </li>
      <li data-it>None</li>
    </ul>
</details></div>



HTML(<div class="coll">
<details open data-coll-name="a"><summary>a</summary>    <ul>
      <li data-it>abc</li>
      <li data-it>13</li>
      <li data-it>
<details open data-coll-name="d"><summary>d</summary>          <ul>
            <li data-it>efg</li>
            <li data-it>133</li>
          </ul>
</details>      </li>
      <li data-it>None</li>
    </ul>
</details></div>
)

In [ ]:
dtl = DetailsColl({
    'Apollo astronauts': [
        'Neil Armstrong',
        '<a href="/asd">Alan Bean</a>',
        'Buzz Aldrin',
        {
            'letters': [
                'a', 
                'b',
                'c', 
            ]
        },
        'Edgar Mitchell',
        'Alan Shepard'
    ]
})

test_eq(val_at(dtl, 'Apollo astronauts.5'), 'Alan Shepard')
test_eq(val_at(dtl, 'Apollo astronauts.3.letters.0'), 'a')
print(to_xml(dtl))
dtl.show()

<div class="coll">
<details open data-coll-name="Apollo astronauts"><summary>Apollo astronauts</summary>    <ul>
      <li data-it>Neil Armstrong</li>
      <li data-it>&lt;a href="/asd"&gt;Alan Bean&lt;/a&gt;</li>
      <li data-it>Buzz Aldrin</li>
      <li data-it>
<details open data-coll-name="letters"><summary>letters</summary>          <ul>
            <li data-it>a</li>
            <li data-it>b</li>
            <li data-it>c</li>
          </ul>
</details>      </li>
      <li data-it>Edgar Mitchell</li>
      <li data-it>Alan Shepard</li>
    </ul>
</details></div>



HTML(<div class="coll">
<details open data-coll-name="Apollo astronauts"><summary>Apollo astronauts</summary>    <ul>
      <li data-it>Neil Armstrong</li>
      <li data-it>&lt;a href="/asd"&gt;Alan Bean&lt;/a&gt;</li>
      <li data-it>Buzz Aldrin</li>
      <li data-it>
<details open data-coll-name="letters"><summary>letters</summary>          <ul>
            <li data-it>a</li>
            <li data-it>b</li>
            <li data-it>c</li>
          </ul>
</details>      </li>
      <li data-it>Edgar Mitchell</li>
      <li data-it>Alan Shepard</li>
    </ul>
</details></div>
)

In [ ]:
tr = list(dir_tree('.'))
nbs_dtl = dtl = DetailsColl({'nbs': tr})
nbs_dtl.show()

HTML(<div class="coll">
<details open data-coll-name="nbs"><summary>nbs</summary>    <ul>
      <li data-it>00_basic.ipynb</li>
      <li data-it>00_dutil.ipynb</li>
      <li data-it>00_project.ipynb</li>
      <li data-it>01_display.ipynb</li>
      <li data-it>01_display_corrupted.ipynb</li>
      <li data-it>01_flakes.ipynb</li>
      <li data-it>02_git.ipynb</li>
      <li data-it>02_isolated.ipynb</li>
      <li data-it>02_logger.ipynb</li>
      <li data-it>02_logger_loguru.ipynb</li>
      <li data-it>02_server.ipynb</li>
      <li data-it>05_test.ipynb</li>
      <li data-it>10_callback.ipynb</li>
      <li data-it>15_config.ipynb</li>
      <li data-it>17_display.ipynb</li>
      <li data-it>20_widgets.ipynb</li>
      <li data-it>_quarto.yml</li>
      <li data-it>apilist.txt</li>
      <li data-it>
<details open data-coll-name="data"><summary>data</summary>          <ul>
            <li data-it>data/log_across.ipynb</li>
            <li data-it>data/test.ipynb</li>
          </ul>
</details>      </li>
      <li data-it>index.ipynb</li>
      <li data-it>llms-ctx-full.txt</li>
      <li data-it>llms-ctx.txt</li>
      <li data-it>llms.txt</li>
      <li data-it>nbdev.yml</li>
      <li data-it>
<details open data-coll-name="static"><summary>static</summary>          <ul>
            <li data-it>static/apollo_astronauts.json</li>
            <li data-it>static/file.txt</li>
            <li data-it>static/wordlist.txt</li>
          </ul>
</details>      </li>
      <li data-it>styles.css</li>
    </ul>
</details></div>
)

## DetailsJSON
> Collapsible JSON/dict viewer component for notebooks

A FastHTML component that renders dictionaries and nested data as expandable `<details>` elements with syntax highlighting.

In [ ]:
%%HTML
<style>
    details.json ul { list-style-type:none; list-style-position: outside; padding-inline-start: 22px; margin: 0px; }
</style>
<details open class="json">
<summary>Apollo astronauts</summary>
<ul>
  <li><span>1</span>: Neil Armstrong</li>
  <li><span>2</span>: Alan Bean</li>
  <li><details>
<summary>Apollo 11</summary>
<ul>
  <li><span>1</span>: Neil Armstrong</li>
  <li><span>2</span>: Alan Bean</li>
  <li><div><span>3</span>: Buzz Aldrin</div></li>
  <li><span>4</span>: Edgar Mitchell</li>
  <li><span>5</span>: Alan Shepard</li>
</ul></li>
  <li><span>4</span>: Edgar Mitchell</li>
  <li><span>5</span>: Alan Shepard</li>
</ul>

</details>

In [ ]:
#| exporti
def Val(v): 
    "Render value with appropriate CSS class based on type"
    c = (
        'null' if v is None else 
        'true' if v is True else 
        'false' if v is False else 
        'string' if isinstance(v, str) else 
        'number' if isinstance(v, (int, float)) else 
        '')
    return Span(shorten(v, 'r', 140) if v is not None else 'None', cls=f"v {c}")

In [ ]:
#| exporti
def NameVal(k, v):
    "Render key-value pair with name and value styling"
    return Span(Span(k, cls='n'), ': ', Val(v))

In [ ]:
#| export
_css_json = (
    'details.json ul { list-style-type:none; list-style-position: outside; padding-inline-start: 22px; margin: 0; } '
    '''details.json .string { color: #24837b; } details.json .string::before { content: "'"; } details.json .string::after { content: "'"; } '''
    'details.json .number { color: #ad8301; } '
    'details.json .true { color: blue; } '
    'details.json .false { color: red; } '
    'details.json .null { color: gray; } '
    'details.json span.n { color: darkgrey; } '
)


In [ ]:
#| export
class DetailsJSON(dict):
    "Interactive collapsible JSON viewer with HTML details/summary structure"
    def __init__(self, *args, summary:str='', open:bool=True, openall:bool=False, skip:Sequence[str]=(), **kwargs):
        super().__init__(*args, **kwargs)
        self.summary, self.open, self.openall, self.skip = str(summary), open, openall, skip
    def show(self): show(self)
    def __ft__(self, d:Mapping|None=None, summary:str|None=None, lvl:int=0, open:bool=False):
        if d is None: d = self; summary = self.summary or 'summary'; open=self.open
        open = self.openall or open
        return (
            Details(open=open, cls="json")(
                Summary(summary, _n),
                Ul()(*(
                    Li(NameVal(k, v)) if k in self.skip else
                    self.__ft__(v, k, lvl+1) if isinstance(v, Mapping) else
                    self.__ft__(dict(list(zip(range(len(v)), v))), k, lvl+1) if is_listy(v) else
                    Li(NameVal(k, v)) 
                    for k,v in d.items()))))
    _css_ = _css_json
    @classmethod
    def setup_style(cls): add_style(cls._css_)

In [ ]:
#| export
DetailsJSON.setup_style()

In [ ]:
dtl = DetailsJSON({
    '1': 'Neil Armstrong',
    '2': 'Alan Bean',
    '3': 'Buzz Aldrin',
    'letters': {
        'a': 1, 
        'b': 2
        },
    '5': 'Edgar Mitchell',
    '6': 'Alan Shepard'
}, summary='Apollo astronauts')

test_eq(val_at(dtl, '5'), 'Edgar Mitchell')
test_eq(val_at(dtl, 'letters.a'), 1)
print(to_xml(dtl))
dtl.show()

In [ ]:
d= {
    "idx": 1,
    "cell_type": "code",
    "source": "# cell 1\nprint('hello')",
    "id": "W1sZmlsZQ==",
    "metadata": {
        "brd": {
            "id": "717322f8-95fa-425c-839d-8b9e7d4ef921"
        }
    },
    "outputs": [
        {'output_type': 'stream', 'name': 'stdout', 'text': '1\n'},
        {'output_type': 'stream', 'name': 'stdout', 'text': '2\n'}
    ],
    "execution_count": 1
}
DetailsJSON(d, openall=True).show()

In [ ]:
request = {
    'headers': {
        'HX-Request': 'true',
        'HX-Current-URL': 'vscode-webview://1ql27...enderer'
    },
    'headerNames': {
        'hx-request': 'HX-Request',
        'hx-current-url': 'HX-Current-URL'
    },
    'status': 0,
    'method': 'GET',
    'url': '/DetailsJSON_5096628128/Apollo 11/Buzz Aldrin',
    'async': True,
    'timeout': 0,
    'withCredentials': False,
    'body': None,
    'req_id': '68ffadb0-958d-4346-b314-d9d62ca247d7'
}

response = {
    'headers': {
        'content-length': '441',
        'content-type': 'text/html; charset=utf-8',
        'last-modified': 'Fri, 15 Nov 2024 16:22:15 GMT',
        'cache-control': 'no-store, no-cache, must-revalidate'
    },
    'status': 200,
    'statusText': 'OK',
    'data': '<details open><summary>Buzz Aldrin</summary>\n   <ul>\n     '
'<li>Pilot on Gemini 12 and Lunar Module pilot on Apollo 11.</li>\n     '
'<li>Aldrin was the second person to walk on the moon.</li>\n     <li>The maiden '
'name of Aldrin&#x27;s mother was &quot;Moon.&quot;</li>\n     <li>While Neil was '
'the first human to step onto the moon, I&#x27;m the first alien from another '
'world to enter a spacecraft that was going to Earth.</li>\n   '
'</ul>\n</details>',
    'xml': None,
    'finalUrl': 'http://nb/DetailsJSON_5096628128/Apollo%2011/Buzz%20Aldrin',
    'req_id': '68ffadb0-958d-4346-b314-d9d62ca247d7'
}

In [ ]:
req = DetailsJSON(request, summary='request', open='all')
req.show()

In [ ]:
resp = DetailsJSON(response, summary='response')
resp.show()

# export -

In [ ]:
from pote.flakes import show_flakes
await show_flakes()

<div class="prose">

No warnings to report

</div>

In [ ]:
# #|hide
# #|eval: false
# from pote.dialog import dlg_export
# await dlg_export()